# final_01 练习：全流程综合题

与讲解版逐 Cell 对应。整条流水线被挖空，`assert` 验收**保留**——
跑通所有 `assert` 即交付通过。卡住就回讲解版看对应阶段。

## 一、考点回顾

| 阶段 | 覆盖章节 | 核心动作 |
|---|---|---|
| 读取 | ch02 / ch06 | 编码、na_values、dtype 对账 |
| 清洗 | ch06 / ch05 | 字符串转数值、日期、类别标准化、去重 |
| 补全 | ch04 / ch05 / ch11 | 哨兵置 NaN、clip、分组中位数 transform 填充 |
| 合并 | ch13 | left join + 缺键排查 |
| 分析 | ch07 / ch09 / ch10 | 聚合画像、分箱、风险分级 |
| 交付 | ch18 | 三张图 + 报告 |

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

# 中文显示：字体列表按平台命中率从高到低排；负号必须显式修复
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB",
                                   "Arial Unicode MS", "SimHei", "Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False

DATA = Path("../data")

print("pandas", pd.__version__)

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def probe(fn, *args, **kwargs):
    """把「调用可能抛异常」写成返回值，避免在挖空块里写 try/except。

    返回 ("ok", 结果) 或 ("err", 异常类型名)。
"""
    try:
        return "ok", fn(*args, **kwargs)
    except Exception as exc:
        return "err", type(exc).__name__


def close_all():
    """每次画完关掉图窗，避免内存里堆几十张图。"""
    plt.close("all")


def png_bytes(fig):
    """把图渲染成 PNG 字节流（不写文件即可验证 savefig 真的产出了图）。"""
    import io

    buf = io.BytesIO()
    fig.savefig(buf, dpi=110, bbox_inches="tight")
    return buf.getvalue()


def is_png(data):
    """PNG 魔数校验：b"\\x89PNG"。"""
    return data[:4] == b"\x89PNG"


print("脚手架就绪：probe / png_bytes / is_png / close_all")

## 题 1：读取与编码对账（对应讲解 阶段 0）

In [ ]:
status, msg = probe(pd.read_csv, DATA / "device_defects_gbk.csv")

gbk = pd.read_csv(DATA / "device_defects_gbk.csv", encoding="gbk")
raw = pd.read_csv(DATA / "device_defects_raw.csv", encoding="utf-8-sig")

print("utf-8 读 GBK 文件 →", status, "|", msg)
print("gbk 编码读取成功:", gbk.shape)
print("raw:", raw.shape, "| 负荷值 dtype:", raw["负荷值"].dtype)

# 负荷值列缺失对账：自动识别 14 个 vs 漏网的 "—" 7 个 = 真缺失 21 个
print("自动识别缺失:", int(raw["负荷值"].isna().sum()),
      "| 漏网的 -- :", int((raw["负荷值"] == "—").sum()),
      "| 含 kW:", int(raw["负荷值"].str.contains("kW").sum()),
      "| 含千分位:", int(raw["负荷值"].str.contains(",").sum()))

assert status == "err" and msg == "UnicodeDecodeError", "utf-8 读 GBK 必须报错"
assert gbk.shape == (270, 13)
assert raw.shape == (270, 13)
assert pd.api.types.is_string_dtype(raw["负荷值"]), "raw 的负荷值还是字符串"
assert int(raw["负荷值"].isna().sum()) == 14, "N/A 与空串被自动识别"
assert int((raw["负荷值"] == "—").sum()) == 7, "-- 不会被自动识别"
assert int(raw["负荷值"].str.contains("kW").sum()) == 5
assert int(raw["负荷值"].str.contains(",").sum()) == 7

**为什么这样做**：读脏数据先做缺失对账——14 自动识别 + 7 个 `—` = 21，
这组数字后面每一步都要用到。

## 题 2：负荷值从字符串到数值（对应讲解 阶段 1）

In [ ]:
load = (raw["负荷值"].astype(str)
        .str.replace(" kW", "", regex=False)
        .str.replace(",", "", regex=False))
df = raw.assign(负荷值=pd.to_numeric(load, errors="coerce"))

print("清洗后 dtype:", df["负荷值"].dtype, "| 缺失:", int(df["负荷值"].isna().sum()))
print(">=1000:", int((df["负荷值"] >= 1000).sum()), "(9999 x4 + 5000 x3)",
      "| <0:", int((df["负荷值"] < 0).sum()), "(-1 x4)")

assert df["负荷值"].dtype == np.dtype("float64")
assert int(df["负荷值"].isna().sum()) == 21, "14 自动识别 + 7 个 -- 全部落水"
assert int((df["负荷值"] >= 1000).sum()) == 7
assert int((df["负荷值"] == 9999).sum()) == 4
assert int((df["负荷值"] == 5000).sum()) == 3
assert int((df["负荷值"] == -1).sum()) == 4

## 题 3：日期 4 格式统一（对应讲解 阶段 2）

In [ ]:
d = (df["发现日期"].astype(str)
     .str.replace("年", "-", regex=False)
     .str.replace("月", "-", regex=False)
     .str.replace("日", "", regex=False))
df["发现日期"] = pd.to_datetime(d, format="mixed")

print("NaT:", int(df["发现日期"].isna().sum()),
      "| 范围:", df["发现日期"].min(), "~", df["发现日期"].max())

assert int(df["发现日期"].isna().sum()) == 0, "4 种格式全部解析成功"
assert df["发现日期"].min() == pd.Timestamp("2026-01-02")
assert df["发现日期"].max() == pd.Timestamp("2026-06-30")

## 题 4：类别标准化 8->3、10->6（对应讲解 阶段 3）

In [ ]:
df["处理状态"] = (df["处理状态"].astype(str).str.strip()
                 .replace({"已 处理": "已处理", "处理完成": "已处理", "done": "已处理",
                           "处理中 ": "处理中", "待处理": "未处理"}))

print("状态类别:", df["处理状态"].nunique(),
      df["处理状态"].value_counts().to_dict())

df["缺陷类型"] = df["缺陷类型"].astype(str).str.strip()
df["缺陷类型"] = df["缺陷类型"].replace({"渗油": "渗漏油", "漏油": "渗漏油", "异物搭挂": "异物"})

print("缺陷类别:", df["缺陷类型"].nunique(),
      df["缺陷类型"].value_counts().to_dict())

assert df["处理状态"].nunique() == 3
assert int((df["处理状态"] == "已处理").sum()) == 171
assert int((df["处理状态"] == "处理中").sum()) == 54
assert int((df["处理状态"] == "未处理").sum()) == 45
assert df["缺陷类型"].nunique() == 6
assert int((df["缺陷类型"] == "渗漏油").sum()) == 61
assert int((df["缺陷类型"] == "异物").sum()) == 40

## 题 5：去重两连击（对应讲解 阶段 4）

In [ ]:
d1 = df.drop_duplicates()
print("全列去重:", d1.shape)

# 业务键：记录ID可以不同，其余 7 列相同就是同一条业务记录
BIZ_KEY = ["台区编号", "线路名称", "设备类型", "设备编号", "缺陷类型", "缺陷等级", "发现日期"]

d2 = d1.drop_duplicates(subset=BIZ_KEY).reset_index(drop=True)
print("业务键去重:", d2.shape)

assert d1.shape == (264, 13), "完全重复 6 行"
assert d2.shape == (260, 13), "业务键重复再删 4 行"
assert d2.index.tolist() == list(range(260)), "reset_index 让后续索引连续"

**为什么这样做**：先全列去重再业务键去重，顺序反了会把「记录ID不同
但内容相同」的行留下来——业务键才是判断重复的业务口径。

## 题 6：哨兵、越界与分组填充（对应讲解 阶段 5）

In [ ]:
# 哨兵先记账：负荷值 >=1000 或 <0 的行数
n_sent = int(((d2["负荷值"] >= 1000) | (d2["负荷值"] < 0)).sum())

d2["负荷值"] = d2["负荷值"].mask((d2["负荷值"] >= 1000) | (d2["负荷值"] < 0))
n_load_na = int(d2["负荷值"].isna().sum())
print("哨兵值:", n_sent, "| 置 NaN 后缺失:", n_load_na)

# 湿度越界先记账：物理范围 [0, 100] 之外共 5 个
n_hum = int(((d2["湿度"] > 100) | (d2["湿度"] < 0)).sum())

d2["湿度"] = d2["湿度"].clip(0, 100)
print("湿度越界:", n_hum, "| clip 后越界:", int(((d2["湿度"] > 100) | (d2["湿度"] < 0)).sum()))

# 温度：MAR 缺失（仅 C/D 台区），用全局中位数填充
temp_med = float(d2["温度"].median())

d2["温度"] = d2["温度"].fillna(temp_med)
print("温度缺失 23 -> 中位数填充:", temp_med)

grp_med = d2.groupby("设备类型")["负荷值"].transform("median")
d2["负荷值"] = d2["负荷值"].fillna(grp_med)
print("填充后缺失:", int(d2["负荷值"].isna().sum()))
print("填充后 mean:", round(float(d2["负荷值"].mean()), 4),
      "| median:", round(float(d2["负荷值"].median()), 4),
      "| max:", round(float(d2["负荷值"].max()), 4))

assert n_sent == 11 and n_load_na == 32, "21 原缺失 + 11 哨兵"
assert n_hum == 5 and int(((d2["湿度"] > 100) | (d2["湿度"] < 0)).sum()) == 0
assert temp_med == 21.9
assert int(d2["温度"].isna().sum()) == 0
assert int(d2["负荷值"].isna().sum()) == 0
assert round(float(d2["负荷值"].mean()), 4) == 175.3413
assert round(float(d2["负荷值"].median()), 4) == 88.695
assert round(float(d2["负荷值"].max()), 4) == 588.45, "截断后最大值是尖峰填充的结果"

## 题 7：合并台账与缺键排查（对应讲解 阶段 6）

In [ ]:
info = pd.read_csv(DATA / "device_info.csv")
dfm = d2.merge(info, on="台区编号", how="left")

# 先记账再填充：缺键不报错不警告，只是安静地 NaN
n_info_na = int(dfm["所属供电所"].isna().sum())
na_stations = sorted(dfm.loc[dfm["所属供电所"].isna(), "台区编号"].unique().tolist())
print("info:", info.shape, "| merge 后:", dfm.shape,
      "| 所属供电所缺失:", n_info_na)
print("缺失台区:", na_stations)

dfm["所属供电所"] = dfm["所属供电所"].fillna("未知")
print("填充后各供电所记录数:", dfm["所属供电所"].value_counts().to_dict())

assert info.shape == (16, 5)
assert dfm.shape == (260, 17), "13 + 5 台账列 - 1 连接键 = 17"
assert n_info_na == 23, "不报错不警告，只是安静地 NaN"
assert na_stations == ["STATION_B_03", "STATION_F_02"]
assert int((dfm["所属供电所"] == "未知").sum()) == 23

## 题 8：聚合画像（对应讲解 阶段 7）

In [ ]:
pt = dfm.pivot_table(index="设备类型", columns="缺陷等级", values="负荷值", aggfunc="mean")
print("pivot:", pt.shape)
print(pt.round(4))

rep = (dfm.groupby("设备类型")
       .agg(记录数=("记录ID", "count"),
            危急数=("缺陷等级", lambda s: int((s == "危急").sum())),
            平均负荷=("负荷值", "mean")))
rep["危急占比"] = rep["危急数"] / rep["记录数"]
print(rep.round(4))

done_rate = float((dfm["处理状态"] == "已处理").mean())
print("全局已处理率:", round(done_rate, 4))

assert pt.shape == (5, 3), "5 种设备 x 3 个等级"
assert round(float(pt.loc["变压器", "危急"]), 4) == 528.394
assert rep.shape == (5, 4)
assert int(rep["危急数"].sum()) == 30, "全表危急 30 条"
assert int(rep.loc["避雷器", "危急数"]) == 12, "避雷器危急最多"
assert round(float(rep.loc["避雷器", "危急占比"]), 4) == 0.2143
assert round(done_rate, 4) == 0.6308

## 题 9：分箱与风险分级（对应讲解 阶段 8）

In [ ]:
bins = pd.cut(dfm["负荷值"], [0, 50, 150, 300, 1000], labels=["极低", "低", "中", "高"])
print("cut 计数:", bins.value_counts().reindex(["极低", "低", "中", "高"]).to_dict(),
      "| NaN:", int(bins.isna().sum()))

risk = np.select([dfm["负荷值"] > 300, dfm["负荷值"] > 150],
                 ["高风险", "中风险"], default="低风险")
dfm["风险等级"] = risk
print("风险分级:", pd.Series(risk).value_counts().to_dict())

ct = pd.crosstab(dfm["设备类型"], dfm["风险等级"])
print("crosstab:", ct.shape)
print(ct)

assert int(bins.isna().sum()) == 0, "负荷值已填充，分箱不应再有 NaN"
assert bins.value_counts().reindex(["极低", "低", "中", "高"]).tolist() == [62, 109, 36, 53]
assert pd.Series(risk).value_counts().to_dict() == {"低风险": 171, "高风险": 53, "中风险": 36}
assert int(((dfm["风险等级"] == "高风险") & (dfm["缺陷等级"] == "危急")).sum()) == 5
assert ct.shape == (5, 3)

## 题 10：可视化三件套（对应讲解 阶段 9）

In [ ]:
# 图 1：各设备类型危急缺陷数（柱状）
fig1, ax1 = plt.subplots(figsize=(6, 3.5))

rep["危急数"].plot(kind="bar", ax=ax1, title="各设备类型危急缺陷数")

fig1.tight_layout()
data1 = png_bytes(fig1)
print("bar 图 PNG 字节数:", len(data1), "| 魔数合法:", is_png(data1))
close_all()

assert is_png(data1), "savefig 真的产出了 PNG"

In [ ]:
# 图 2：负荷值分布（直方图，range 压住长尾）
fig2, ax2 = plt.subplots(figsize=(6, 3.5))

dfm["负荷值"].plot(kind="hist", bins=20, range=(0, 1000), ax=ax2, title="清洗后负荷值分布")

fig2.tight_layout()
counts, _ = np.histogram(dfm["负荷值"], bins=20, range=(0, 1000))
print("hist 20 柱总频数:", int(counts.sum()), "← 必须等于行数，一根都不能丢")
close_all()

assert int(counts.sum()) == len(dfm) == 260, "range 内数据一行不丢"
assert is_png(png_bytes(fig2))
close_all()

In [ ]:
# 图 3：台区 A_01 的日均负荷走势（时序线图）
load_curve = pd.read_csv(DATA / "load_curve.csv", parse_dates=["时间戳"])
st = (load_curve[load_curve["台区编号"] == "STATION_A_01"]
      .sort_values("时间戳").set_index("时间戳"))

daily = st["负荷值"].resample("D").mean()
print("daily:", daily.shape, "| NaN:", int(daily.isna().sum()),
      "| min:", round(float(daily.min()), 2), "| max:", round(float(daily.max()), 2))

# 节假日 / 工作日的时段均值，交叉验证时序规律
hol_mean = float(st.loc[st["是否节假日"], "负荷值"].mean())
work_mean = float(st.loc[~st["是否节假日"], "负荷值"].mean())
print("节假日日均:", round(hol_mean, 3), "| 工作日日均:", round(work_mean, 3))

fig3, ax3 = plt.subplots(figsize=(8, 3))

daily.plot(ax=ax3, title="STATION_A_01 日均负荷走势")

fig3.tight_layout()
assert is_png(png_bytes(fig3))
close_all()

assert daily.shape == (90,) and int(daily.isna().sum()) == 0
assert round(float(daily.min()), 2) == 428.33
assert round(float(daily.max()), 2) == 784.83
assert round(hol_mean, 3) == 513.516 and round(work_mean, 3) == 678.735

**为什么这样做**：每张图都过 `png_bytes` + PNG 魔数验证，hist 做频数
对账（260 行一根不丢），line 用节假日 / 工作日均值交叉验证时序规律。

## 题 11：最终报告（对应讲解 阶段 10）

In [ ]:
report = pd.DataFrame({
    "指标": ["原始行数", "清洗后行数", "负荷值缺失(原始)", "哨兵值", "日期解析成功",
             "状态类别(前/后)", "缺陷类别(前/后)", "危急记录", "高风险记录", "台账缺失台区"],
    "数值": [270, len(dfm), 21, n_sent, int(dfm["发现日期"].notna().sum()),
             "8 -> 3", "10 -> 6", int((dfm["缺陷等级"] == "危急").sum()),
             int((dfm["风险等级"] == "高风险").sum()), 2],
})
print(report.to_string(index=False))

assert report["数值"].tolist() \
    == [270, 260, 21, 11, 260, "8 -> 3", "10 -> 6", 30, 53, 2]
print("全流程交付完成：raw(270,13) -> dfm(260,18) + 三张图 + 一份报告")

---

## 综合自查

1. 题 1 的缺失对账数字（14 + 7）如果对不上，后面哪几个 assert 会连锁崩？
2. 题 6 为什么必须先 mask 哨兵再填充？直接填充会怎样？
3. 题 7 的 23 行缺失是怎么算出来的？（提示：不是 2）
4. 题 9 的 `cut` NaN 必须为 0——它在替你验收前面哪一步？
5. 题 11 报告里的「高风险记录 53」和「危急记录 30」为什么不相等？

全部对得上账，pandas 方向 19/19 交付完成。